# Papua: are the results driven by the Papua conflict?

Papua (GADM provinces Papua and Papua Barat) differs from the rest of Indonesia: it has an active separatist insurgency, much of its violence against civilians is carried out by armed groups rather than by state forces, and fire activity there is concentrated in the south. `mechanisms.ipynb` found that violence against civilians by non-state actors, mostly in Papua, falls with upwind fire by a similar percentage to that by state forces, which neither the smoke nor the state-capacity channel obviously explains.

This notebook asks two questions:

1. **Do the main results and mechanism tests hold outside Papua?** Re-estimate the reduced form, the mechanism decomposition and the same/other-province split excluding Papua.
2. **What happens inside Papua?** Estimate the same outcomes on Papua alone. With only two provinces, standard errors there are clustered by district rather than province.

Specifications follow `mechanisms.ipynb` and `state-capacity-province.ipynb`: reduced form with log upwind FRP and its one-month lag, district and year-month fixed effects.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import pyfixest as pf
from scipy import stats
warnings.filterwarnings('ignore')

ROOT         = str(Path.cwd().resolve().parent)  # repository root; run notebooks from analysis/
PANEL_CSV = f'{ROOT}/analysis/iv_panel_district.csv'
INSTR_PROV = f'{ROOT}/analysis/output/upwind_instrument_by_province.csv'
DISTRICT_SHP = f'{ROOT}/data/administrative/gadm41_IDN_shp/gadm41_IDN_2.shp'
ACLED_XLSX = f'{ROOT}/data/unrest/ACLED Data_2026-06-10.xlsx'
OUT_CSV = f'{ROOT}/analysis/output/papua_results.csv'
INSTR = ['log_upwind_frp', 'log_upwind_frp_l1']
FE = 'district + year_month'
panel = pd.read_csv(PANEL_CSV)

## 1. Outcomes (as in `mechanisms.ipynb`) and the province-split instrument

In [2]:
gadm = gpd.read_file(DISTRICT_SHP)[['NAME_1', 'NAME_2', 'geometry']].rename(columns={'NAME_1': 'province', 'NAME_2': 'district'})
dup = gadm['district'].duplicated(keep=False)
gadm.loc[dup, 'district'] = gadm.loc[dup, 'district'] + ' (' + gadm.loc[dup, 'province'] + ')'
acled = pd.read_excel(ACLED_XLSX)
acled['event_date'] = pd.to_datetime(acled['event_date'])
acled['year'], acled['month'] = acled['event_date'].dt.year, acled['event_date'].dt.month
ag = gpd.GeoDataFrame(acled, geometry=gpd.points_from_xy(acled['longitude'], acled['latitude']), crs='EPSG:4326')
j = gpd.sjoin(ag, gadm[['district', 'geometry']], how='left', predicate='within')
j = j[~j.index.duplicated(keep='first')]
acled['district'] = j['district'].reindex(acled.index).values
off = acled['district'].isna()
near = gpd.sjoin_nearest(ag.loc[off].to_crs(3395), gadm[['district', 'geometry']].to_crs(3395), how='left', max_distance=20_000)
near = near[~near.index.duplicated(keep='first')]
acled.loc[off, 'district'] = near['district'].reindex(acled.index[off]).values
et, se, i1 = acled['event_type'], acled['sub_event_type'], acled['inter1'].fillna('')
state = i1.eq('State forces')
O = {'all_events': et.notna(), 'peaceful_protest': se.eq('Peaceful protest'), 'state_forces': state,
     'vac': et.eq('Violence against civilians'), 'vac_state': et.eq('Violence against civilians') & state,
     'vac_nonstate': et.eq('Violence against civilians') & ~state, 'vac_rebel': et.eq('Violence against civilians') & i1.eq('Rebel group'),
     'battles': et.eq('Battles'), 'nonstate_violence': et.isin(['Riots', 'Violence against civilians', 'Battles']) & ~state}
for k, v in O.items():
    acled[k] = v.astype(int)
cnt = acled.groupby(['district', 'year', 'month'])[list(O)].sum().reset_index()
panel = panel.merge(cnt, on=['district', 'year', 'month'], how='left')
panel[list(O)] = panel[list(O)].fillna(0)

inst = pd.read_csv(INSTR_PROV)[['district', 'year', 'month', 'log_same', 'log_other']]
panel = panel.merge(inst, on=['district', 'year', 'month'], how='left').sort_values(['district', 'year', 'month'])
for c in ['log_same', 'log_other']:
    panel[c + '_l1'] = panel.groupby('district')[c].shift(1)

panel['papua'] = panel['province'].isin(['Papua', 'Papua Barat']).astype(int)
share = panel.groupby('district')['events'].apply(lambda s: (s > 0).mean())
active = set(share[share >= 0.30].index)
panel['active'] = panel['district'].isin(active).astype(int)

acled['papua'] = acled['district'].map(panel.drop_duplicates('district').set_index('district')['papua'])
print('Papua districts:', panel.loc[panel.papua == 1, 'district'].nunique(), '| of which conflict-active:',
      panel.loc[(panel.papua == 1) & (panel.active == 1), 'district'].nunique(), 'of', len(active))
print('\nShare of each outcome occurring in Papua:')
print((acled.groupby('papua')[list(O)].sum().T.assign(share_papua=lambda t: t[1] / (t[0] + t[1]))['share_papua']).round(3).to_string())
print('\nVAC by non-state actors in Papua, by actor type:')
print(acled[(acled.papua == 1) & (acled.vac_nonstate == 1)]['inter1'].value_counts().to_string())

Papua districts: 39 | of which conflict-active: 15 of 85

Share of each outcome occurring in Papua:
all_events           0.170
peaceful_protest     0.092
state_forces         0.617
vac                  0.629
vac_state            0.648
vac_nonstate         0.603
vac_rebel            0.945
battles              0.790
nonstate_violence    0.378

VAC by non-state actors in Papua, by actor type:
inter1
Rebel group          206
Political militia     61


## 2. Estimates

Samples: conflict-active districts (i) all, (ii) excluding Papua, (iii) Papua only; and the full panel excluding Papua. Within Papua, all its districts are used (not only conflict-active ones), because only a few are conflict-active. For each, OLS and PPML on contemporaneous log upwind FRP, plus the same/other-province split.

In [3]:
def ppml(y, d, rhs, vcov):
    d = d[d.groupby('district')[y].transform('sum') > 0]
    if d[y].sum() < 20 or d['district'].nunique() < 3:
        return None
    try:
        return pf.fepois(f'{y} ~ {rhs} | {FE}', data=d, vcov=vcov)
    except Exception:
        return None

def get(m, t):
    if m is None or t not in m.coef().index:
        return (np.nan, np.nan, np.nan)
    tt = m.tidy().loc[t]
    return tt['Estimate'], tt['Std. Error'], tt['Pr(>|t|)']

SAMPLES = {
    'active, all': (panel[panel.active == 1], {'CRV1': 'province'}),
    'active, excl. Papua': (panel[(panel.active == 1) & (panel.papua == 0)], {'CRV1': 'province'}),
    'full panel, excl. Papua': (panel[panel.papua == 0], {'CRV1': 'province'}),
    'Papua only (all districts)': (panel[panel.papua == 1], {'CRV1': 'district'}),
}
rows = []
for sname, (d, vc) in SAMPLES.items():
    print(f"\n=== {sname}: {d['district'].nunique()} districts, clustered by {vc['CRV1']} ===")
    for y in O:
        r = {'sample': sname, 'outcome': y, 'mean': d[y].mean(), 'n_events': d[y].sum()}
        m = pf.feols(f'{y} ~ {" + ".join(INSTR)} | {FE}', data=d, vcov=vc)
        r['ols'], r['ols_se'], r['ols_p'] = get(m, 'log_upwind_frp')
        r['ols_pct_mean'] = 100 * r['ols'] / r['mean'] if r['mean'] > 0 else np.nan
        p = ppml(y, d, ' + '.join(INSTR), vc)
        r['ppml'], r['ppml_se'], r['ppml_p'] = get(p, 'log_upwind_frp')
        dd = d.dropna(subset=['log_same_l1', 'log_other_l1'])
        p2 = ppml(y, dd, 'log_same + log_other + log_same_l1 + log_other_l1', vc)
        r['ppml_same'], r['ppml_same_se'], r['ppml_same_p'] = get(p2, 'log_same')
        r['ppml_other'], r['ppml_other_se'], r['ppml_other_p'] = get(p2, 'log_other')
        rows.append(r)
        print(f"  {y:18s} events {int(r['n_events']):6d} | OLS {r['ols']:+.4f} (p={r['ols_p']:.3f}) | PPML {r['ppml']:+.3f} (p={r['ppml_p']:.3f})"
              f" | same {r['ppml_same']:+.3f} (p={r['ppml_same_p']:.3f}) other {r['ppml_other']:+.3f} (p={r['ppml_other_p']:.3f})")
res = pd.DataFrame(rows)
res.to_csv(OUT_CSV, index=False)
print(f'\nSaved {len(res)} rows to {OUT_CSV}')


=== active, all: 85 districts, clustered by province ===


  all_events         events  14848 | OLS -0.0261 (p=0.002) | PPML -0.013 (p=0.030) | same -0.004 (p=0.614) other -0.013 (p=0.011)
  peaceful_protest   events  11441 | OLS -0.0192 (p=0.003) | PPML -0.014 (p=0.017) | same -0.003 (p=0.718) other -0.018 (p=0.001)
  state_forces       events    722 | OLS -0.0057 (p=0.038) | PPML -0.063 (p=0.000) | same -0.045 (p=0.009) other -0.038 (p=0.377)


  vac                events    764 | OLS -0.0055 (p=0.068) | PPML -0.067 (p=0.000) | same -0.064 (p=0.000) other -0.028 (p=0.413)
  vac_state          events    460 | OLS -0.0035 (p=0.062) | PPML -0.075 (p=0.000) | same -0.068 (p=0.000) other -0.048 (p=0.308)
  vac_nonstate       events    304 | OLS -0.0020 (p=0.158) | PPML -0.073 (p=0.008) | same -0.073 (p=0.022) other +0.001 (p=0.980)


  vac_rebel          events    176 | OLS -0.0002 (p=0.586) | PPML -0.048 (p=0.000) | same -0.047 (p=0.000) other -0.011 (p=0.965)
  battles            events    468 | OLS +0.0002 (p=0.791) | PPML -0.024 (p=0.172) | same -0.011 (p=0.616) other +0.052 (p=0.559)
  nonstate_violence  events   1737 | OLS -0.0039 (p=0.128) | PPML -0.010 (p=0.513) | same -0.001 (p=0.961) other +0.005 (p=0.796)

=== active, excl. Papua: 70 districts, clustered by province ===


  all_events         events  11729 | OLS -0.0199 (p=0.041) | PPML -0.008 (p=0.212) | same +0.002 (p=0.749) other -0.014 (p=0.007)
  peaceful_protest   events  10099 | OLS -0.0167 (p=0.060) | PPML -0.011 (p=0.114) | same +0.001 (p=0.875) other -0.017 (p=0.002)
  state_forces       events    213 | OLS -0.0040 (p=0.037) | PPML -0.100 (p=0.006) | same -0.069 (p=0.055) other -0.028 (p=0.506)


  vac                events    203 | OLS -0.0031 (p=0.111) | PPML -0.106 (p=0.039) | same -0.096 (p=0.061) other -0.027 (p=0.517)
  vac_state          events    125 | OLS -0.0027 (p=0.083) | PPML -0.122 (p=0.013) | same -0.121 (p=0.020) other -0.054 (p=0.328)
  vac_nonstate       events     78 | OLS -0.0005 (p=0.556) | PPML -0.055 (p=0.502) | same -0.033 (p=0.733) other +0.014 (p=0.840)
  vac_rebel          events      1 | OLS +0.0002 (p=0.333) | PPML +nan (p=nan) | same +nan (p=nan) other +nan (p=nan)


... (6 outputs left out)

  nonstate_violence  events   1703 | OLS -0.0003 (p=0.562) | PPML -0.000 (p=0.976) | same +0.024 (p=0.068) other -0.013 (p=0.250)

=== Papua only (all districts): 39 districts, clustered by district ===
  all_events         events   3677 | OLS -0.0167 (p=0.222) | PPML -0.018 (p=0.216) | same -0.016 (p=0.288) other -0.002 (p=0.928)
  peaceful_protest   events   1545 | OLS -0.0084 (p=0.313) | PPML -0.009 (p=0.588) | same +0.004 (p=0.817) other -0.042 (p=0.096)


  state_forces       events    621 | OLS -0.0053 (p=0.442) | PPML -0.037 (p=0.478) | same -0.024 (p=0.664) other -0.147 (p=0.034)
  vac                events    674 | OLS -0.0056 (p=0.241) | PPML -0.043 (p=0.224) | same -0.033 (p=0.350) other -0.133 (p=0.029)


  vac_state          events    408 | OLS -0.0029 (p=0.525) | PPML -0.034 (p=0.546) | same -0.019 (p=0.748) other -0.144 (p=0.091)
  vac_nonstate       events    266 | OLS -0.0027 (p=0.089) | PPML -0.077 (p=0.021) | same -0.077 (p=0.026) other -0.092 (p=0.539)
  vac_rebel          events    205 | OLS +0.0004 (p=0.750) | PPML -0.011 (p=0.806) | same -0.007 (p=0.873) other -0.221 (p=0.395)
  battles            events    490 | OLS +0.0011 (p=0.674) | PPML -0.000 (p=0.999) | same -0.003 (p=0.910) other +0.008 (p=0.936)


  nonstate_violence  events   1035 | OLS -0.0024 (p=0.517) | PPML -0.022 (p=0.247) | same -0.031 (p=0.112) other +0.095 (p=0.052)

Saved 36 rows to /Users/hariaksha/Documents/GitHub/climate-conflict/analysis/output/papua_results.csv


## 3. Weak-instrument-robust check of the main result excluding Papua

Anderson–Rubin test (joint Wald test that both instrument coefficients in the reduced form are zero, F(2, G−1)) for all events in the conflict-active sample excluding Papua.

In [4]:
d = panel[(panel.active == 1) & (panel.papua == 0)]
m = pf.feols(f'all_events ~ {" + ".join(INSTR)} | {FE}', data=d, vcov={'CRV1': 'province'})
names = list(m.coef().index); idx = [names.index(t) for t in INSTR]
b = m.coef()[INSTR].to_numpy(); V = m._vcov[np.ix_(idx, idx)]
W = float(b @ np.linalg.solve(V, b)); G = d['province'].nunique()
print(f'Active excl. Papua: {d.district.nunique()} districts, {G} provinces; AR p = {stats.f.sf(W / 2, 2, G - 1):.4f}')

Active excl. Papua: 70 districts, 31 provinces; AR p = 0.0805
